# 1. Основной код поиска совпадений между пролетом спутника и областью грозового кластера для данных SABER

In [6]:
# Подключение библиотек
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import datetime as dt
import os

In [7]:
# Входные данные
file_year = 2013 # Год файла
interval = 15 # 15 минутный интервал вокруг грозового разряда и вокруг момента совпадения

thunderbolts_npz_in = f'C:/Users/Maks/Desktop/Jupyter/thunderbolts_data/npz/{file_year}_thunderbolts_clastered.npz'
thunderbolts_hdf_out = f'C:/Users/Maks/Desktop/Jupyter/thunderbolts_data/hdf/{file_year}_thunderbolts_clastered.h5'

base_satellite_directory = f'G:/SABER_L2A/{file_year}' 
output_filtered_satellite_dir = f'C:/Users/Maks/Desktop/Jupyter/output_filtered_saber_data/{file_year}' 

output_full_saber_data_dir = "C:/Users/Maks/Desktop/Jupyter/output_full_saber_data"

In [8]:
# Преобразование из формата .npz в .hdf, удаление лишних данных, установление даты в качестве индекса
data = np.load(thunderbolts_npz_in)
data_npz = pd.DataFrame(data['strikes']).drop('tail', axis=1).set_index('date')
data_npz.to_hdf(thunderbolts_hdf_out, key='strikes', mode='w', complevel=9)
data_hdf = pd.read_hdf(thunderbolts_hdf_out, 'strikes')

In [4]:
def extract_saber_data(file_path):
    """
    Извлекает ВСЕ данные из SABER-файла: время, координаты, температуру для всех высот
    Параметры:
        file_path: путь к файлу
    Возвращает:
        DataFrame с индексом datetime и колонками: lat, lon, altitude_km, temperature_K
    """
    ds = xr.open_dataset(file_path, decode_timedelta=True)
    
    # Получаем дату из атрибута date (формат YYYYDDD)
    date_value = float(ds['date'].values[0])
    date_str = str(int(date_value))
    year = int(date_str[:4])
    day_of_year = int(date_str[4:])
    base_date = dt.datetime(year, 1, 1) + dt.timedelta(days=day_of_year - 1)
    
    # Создаем список для хранения всех профилей
    all_profiles = []
    
    # Проходим по всем строкам (событиям)
    for row_idx in range(len(ds['ktemp'])):
        # Получаем температуру для текущей строки и удаляем NaN
        temp_series = pd.DataFrame(ds['ktemp']).iloc[row_idx].dropna()
        
        if len(temp_series) > 0:  # Проверяем, есть ли данные
            valid_indices = temp_series.index
            
            # Получаем время для этой строки (в миллисекундах)
            time_ms = pd.DataFrame(ds['time']).iloc[row_idx].loc[valid_indices].values
            
            # Получаем координаты и высоту для валидных индексов
            latitudes = pd.DataFrame(ds['tplatitude']).iloc[row_idx].loc[valid_indices].values
            longitudes = pd.DataFrame(ds['tplongitude']).iloc[row_idx].loc[valid_indices].values
            altitudes = pd.DataFrame(ds['tpaltitude']).iloc[row_idx].loc[valid_indices].values
            temperatures = temp_series.values
            
            # Вычисляем время для каждого измерения
            for i in range(len(valid_indices)):
                # Вычисляем точное время
                current_time = base_date + dt.timedelta(milliseconds=float(time_ms[i]))
                current_time = pd.Timestamp(current_time).floor('s')
                
                all_profiles.append({
                    'datetime': current_time,
                    'lat': latitudes[i],
                    'lon': longitudes[i],
                    'altitude_km': altitudes[i],
                    'temperature_K': temperatures[i]
                })
    
    ds.close()
    
    if len(all_profiles) == 0:
        print(f"Нет данных в файле {file_path}")
        return None
    
    # Создаем DataFrame со всеми данными
    df = pd.DataFrame(all_profiles)
    df = df.set_index('datetime').sort_index()
    
    # Выводим информацию о файле
    print(f"Полный диапазон высот в файле: {df['altitude_km'].min():.2f} - {df['altitude_km'].max():.2f} км")
    print(f"Извлечено {len(df)} записей")
    print(f"Диапазон времени: {df.index.min()} - {df.index.max()}")
    print(f"\n{'='*60}")
    
    return df[['lat', 'lon', 'altitude_km', 'temperature_K']]

In [5]:
# Функция удаления отдаленных разрядов и выделения области наибольшего скопления
def simple_remove_outliers(group, percentile=90, plot=False, cluster_id=None):
    if len(group) <= 5:
        # Для малых кластеров находим самую плотную группу разрядов
        if len(group) == 1:
            # Для одного разряда - минимальная область вокруг него
            lat, lon = group['lat'].iloc[0], group['lon'].iloc[0]
            fixed_span = 0.09  # 10×10 км
            bounds = (
                lat - fixed_span/2,
                lat + fixed_span/2,
                lon - fixed_span/2,
                lon + fixed_span/2
            )
        else:
            # Для 2-5 разрядов находим пару самых близких точек
            coords = group[['lat', 'lon']].values
            min_distance = float('inf')
            closest_pair = None
            # Находим две самые близкие точки
            for i in range(len(coords)):
                for j in range(i+1, len(coords)):
                    distance = np.sqrt((coords[i][0] - coords[j][0])**2 + (coords[i][1] - coords[j][1])**2)
                    if distance < min_distance:
                        min_distance = distance
                        closest_pair = (coords[i], coords[j])
            if closest_pair:
                # Центр между двумя самыми близкими точками
                lat_center = (closest_pair[0][0] + closest_pair[1][0]) / 2
                lon_center = (closest_pair[0][1] + closest_pair[1][1]) / 2
                # Определяем размер области на основе распределения точек
                distances_from_center = np.sqrt(
                    (group['lat'] - lat_center)**2 + (group['lon'] - lon_center)**2
                )
                # Берем 75% перцентиль расстояний + минимальный размер
                radius = max(np.percentile(distances_from_center, 75), 0.045)  # минимум 5 км
                bounds = (
                    lat_center - radius,
                    lat_center + radius,
                    lon_center - radius,
                    lon_center + radius
                )
            else:
                # Запасной вариант
                lat_center = group['lat'].mean()
                lon_center = group['lon'].mean()
                fixed_span = 0.09
                bounds = (
                    lat_center - fixed_span/2,
                    lat_center + fixed_span/2,
                    lon_center - fixed_span/2,
                    lon_center + fixed_span/2
                )
    else:
        # Для больших кластеров - обычная фильтрация по перцентилям
        cut_percent = (100 - percentile) / 2
        lat_min = np.percentile(group['lat'], cut_percent)
        lat_max = np.percentile(group['lat'], 100 - cut_percent)
        lon_min = np.percentile(group['lon'], cut_percent)
        lon_max = np.percentile(group['lon'], 100 - cut_percent)
        bounds = (lat_min, lat_max, lon_min, lon_max)
    if plot:
        plot_cluster_comparison(group, bounds, cluster_id, 
                               "Dense cluster" if len(group) <= 5 else f"Filtered (percentile={percentile}%)")
    return bounds

In [6]:
# Алгоритм по поиску совпадений между пролетом спутника и областью грозового кластера
results = []
# Вычисляем глобальные границы координат для всех грозовых кластеров в файле года
global_lat_min = data_hdf['lat'].min()
global_lat_max = data_hdf['lat'].max()
global_lon_min = data_hdf['lon'].min()
global_lon_max = data_hdf['lon'].max()
# Проходим по всем дням года
for day in range(1, 367): 
    day_str = f"{day:03d}"
    day_dir = os.path.join(base_satellite_directory, day_str)

    if not os.path.exists(day_dir):
        continue
            
    for filename in os.listdir(day_dir):
        if not filename.endswith('.nc'):
            continue
                
        file_path = os.path.join(day_dir, filename)
            
        # Извлекаем данные из SABER-файла с обработкой ошибок
        try:
            dataframe_satellite = extract_saber_data(file_path)
        except Exception as e:
            print(f"Ошибка при обработке {filename}: {e}")
            continue
                
        if dataframe_satellite is None or dataframe_satellite.empty:
            continue

        # Фильтруем по глобальным границам координат гроз
        buffer = 5.0
        mask_coords = (
            (dataframe_satellite['lat'] >= global_lat_min - buffer) & 
            (dataframe_satellite['lat'] <= global_lat_max + buffer) &
            (dataframe_satellite['lon'] >= global_lon_min - buffer) & 
            (dataframe_satellite['lon'] <= global_lon_max + buffer)
        )
        dataframe_satellite = dataframe_satellite[mask_coords]
            
        # Вычисляем временной интервал для файла (из названия файла)
        # Формат: SABER_L2A_2012001_54519_02.07.nc
        year_day = filename.split('_')[2]  # '2012001'
        year = int(year_day[0:4])
        day_of_year = int(year_day[4:7])
        time_start = dt.datetime(year, 1, 1) + dt.timedelta(day_of_year - 1)
        time_end = time_start + dt.timedelta(1)
            
        # Фильтрация файла с данными о грозах data_hdf
        mask = (data_hdf.index >= time_start) & (data_hdf.index <= time_end)
        filtered_hdf = data_hdf[mask]
            
        if not filtered_hdf.empty:
            # Обрабатываем только кластеры с clnb > 0
            clusters = filtered_hdf[filtered_hdf['clnb'] > 0].groupby('clnb')

            sat_time_min = dataframe_satellite.index.min()
            sat_time_max = dataframe_satellite.index.max()
                
            for clnb, group in clusters:
                mask_cluster_time = (
                    (group.index >= sat_time_min - dt.timedelta(minutes=interval)) & 
                    (group.index <= sat_time_max)
                )
                filtered_group = group[mask_cluster_time]
                    
                if filtered_group.empty:
                    continue  # в этом кластере нет разрядов, которые могут совпасть по времени
     
                # Вычисляем границы области скопления
                lat_min, lat_max, lon_min, lon_max = simple_remove_outliers(
                    filtered_group, percentile=90, plot=False
                )
                    
                # Проверяем каждый грозовой разряд в кластере
                for flash_time in filtered_group.index:
                    # Временное окно 15 минут после грозового разряда
                    time_min = flash_time
                    time_max = flash_time + dt.timedelta(minutes=interval)
                        
                    # Поиск совпадений в спутниковых данных
                    mask = (
                        (dataframe_satellite.index >= time_min) & 
                        (dataframe_satellite.index <= time_max) &
                        (dataframe_satellite['lat'] >= lat_min) & 
                        (dataframe_satellite['lat'] <= lat_max) & 
                        (dataframe_satellite['lon'] >= lon_min) & 
                        (dataframe_satellite['lon'] <= lon_max)
                    )
                    matched_data = dataframe_satellite[mask]
                        
                    # Обнаружение и обработка совпадений
                    if not matched_data.empty:
                        first_match = matched_data.iloc[0]
                        match_time = matched_data.index[0]
                        
                        # Находим все разряды в кластере до момента совпадения (включая совпавший)
                        flashes_before_match = group[group.index <= match_time]
                        
                        if len(flashes_before_match) > 0:
                            last_flash_before = flashes_before_match.index.max()  # совпавший разряд
                            
                            # Находим разряды ЗА 15 МИН ДО совпавшего (исключая сам совпавший разряд)
                            time_15min_before = last_flash_before - dt.timedelta(minutes=interval)
                            flashes_in_15min_before = group[
                                (group.index >= time_15min_before) & 
                                (group.index < last_flash_before)  # строго меньше, исключаем совпавший
                            ]
                            
                            flash_times_before = flashes_in_15min_before.index.tolist()
                            
                            # Амплитуда совпавшего разряда 
                            flash_amp = group.loc[last_flash_before, 'amp']
                            
                            # Амплитуды разрядов ЗА 15 МИН ДО совпавшего
                            flash_amps_before = [row['amp'] for idx, row in flashes_in_15min_before.iterrows()]
                            
                            results.append({
                                'clnb': clnb,
                                'satellite_file_name': filename,
                                'altitude_sat': first_match['altitude_km'],
                                'matched_time': match_time,
                                'flash_time': last_flash_before,
                                'time_dif': match_time - last_flash_before,
                                'flash_amp': flash_amp,
                                'flash_times_before': flash_times_before,  # только за 15 мин до
                                'flash_amps_before': flash_amps_before,    # только за 15 мин до
                                # Координаты и параметры спутника
                                'lat_sat': first_match['lat'],   
                                'lon_sat': first_match['lon'],
                                'lat_min': lat_min,
                                'lat_max': lat_max,
                                'lon_min': lon_min,
                                'lon_max': lon_max,
                            })
                                
                            # Сохраняем выделенные данные пролета спутника
                            output_filename = f'trimmed_{filename[:-3]}_clnb_{clnb}.h5'
                            os.makedirs(output_filtered_satellite_dir, exist_ok=True) 
                            output_path = os.path.join(output_filtered_satellite_dir, output_filename)
                                
                            # Фильтруем данные за ±15 минут вокруг момента совпадения
                            trim_time_min = matched_data.index[0] - dt.timedelta(minutes=interval)
                            trim_time_max = matched_data.index[0] + dt.timedelta(minutes=interval)
                            trim_mask = (
                                (dataframe_satellite.index >= trim_time_min) & 
                                (dataframe_satellite.index <= trim_time_max)
                            )
                            trimmed_data = dataframe_satellite[trim_mask]
                                
                            # Сохраняем в HDF
                            trimmed_data.to_hdf(output_path, key='satellite_data', mode='w')
                                
                            break  # Прерываем после первого совпадения для этого кластера
                    
                # Прерываем после первого совпадения для файла? 
                # Если нужно найти все совпадения в файле, уберите break
                # break  

        # Очистка памяти
        del dataframe_satellite

# Создание итогового DataFrame
if results:
    satellite_overpass_matching = pd.DataFrame(results).set_index('clnb').sort_index()
        
    # Сохраняем результат в HDF файл
    os.makedirs(output_full_saber_data_dir, exist_ok=True)
    output_hdf_path = os.path.join(output_full_saber_data_dir, f'{file_year}_saber_overpass_matching.h5')
    
    satellite_overpass_matching.to_hdf(output_hdf_path, key='matches', mode='w', complevel=9)
    
else:
    satellite_overpass_matching = pd.DataFrame(columns=['satellite_file_name', 'flash_time', 'matched_time'])
    print("Совпадений не найдено")
        
# Вывод статистики
print(f"\nРезультаты сопоставления:")
# Подсчет количества файлов SABER
total_saber_files = 0
for day in range(1, 367 if file_year == 2012 else 366):
    day_str = f"{day:03d}"
    day_dir = os.path.join(base_satellite_directory, day_str)
    if os.path.exists(day_dir):
        total_saber_files += len([f for f in os.listdir(day_dir) if f.endswith('.nc')])
print(f"Обработано SABER-файлов: {total_saber_files}")

print(f"Найдено совпадений: {len(satellite_overpass_matching)}")
    
satellite_overpass_matching.T

Полный диапазон высот в файле: 11.78 - 110.00 км
Извлечено 24367 записей
Диапазон времени: 2018-04-20 00:11:49 - 2018-04-20 01:46:28

Полный диапазон высот в файле: 11.63 - 110.00 км
Извлечено 24892 записей
Диапазон времени: 2018-04-20 01:47:11 - 2018-04-20 03:24:47

Полный диапазон высот в файле: 11.50 - 109.99 км
Извлечено 24107 записей
Диапазон времени: 2018-04-20 03:25:12 - 2018-04-20 05:00:54

Полный диапазон высот в файле: 0.00 - 110.00 км
Извлечено 35437 записей
Диапазон времени: 2018-04-20 00:00:00 - 2018-04-20 05:26:45

Полный диапазон высот в файле: 11.74 - 110.00 км
Извлечено 23869 записей
Диапазон времени: 2018-04-20 06:38:40 - 2018-04-20 08:14:21

Полный диапазон высот в файле: 11.61 - 110.00 км
Извлечено 24862 записей
Диапазон времени: 2018-04-20 08:14:50 - 2018-04-20 09:51:42

Полный диапазон высот в файле: 11.75 - 110.00 км
Извлечено 24349 записей
Диапазон времени: 2018-04-20 09:52:07 - 2018-04-20 11:27:50

Полный диапазон высот в файле: 11.37 - 110.00 км
Извлечено 2436

C:\Temp\ipykernel_13412\467516870.py:172: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block3_values] [items->Index(['satellite_file_name', 'flash_times_before', 'flash_amps_before'], dtype='object')]

  satellite_overpass_matching.to_hdf(output_hdf_path, key='matches', mode='w', complevel=9)


Обработано SABER-файлов: 1664
Найдено совпадений: 6


clnb,154,181,388,727,1698,2327
satellite_file_name,SABER_L2A_2018160_89463_02.07.nc,SABER_L2A_2018163_89507_02.07.nc,SABER_L2A_2018176_89707_02.07.nc,SABER_L2A_2018181_89782_02.07.nc,SABER_L2A_2018199_90048_02.07.nc,SABER_L2A_2018215_90285_02.07.nc
altitude_sat,102.0271,107.031693,30.237631,103.061371,100.348053,82.000946
matched_time,2018-06-09 10:52:07,2018-06-12 09:52:53,2018-06-25 20:19:36,2018-06-30 21:28:21,2018-07-18 18:40:16,2018-08-03 17:10:20
flash_time,2018-06-09 10:46:46,2018-06-12 09:50:03,2018-06-25 20:19:06,2018-06-30 21:27:58,2018-07-18 18:33:43,2018-08-03 17:07:37
time_dif,0 days 00:05:21,0 days 00:02:50,0 days 00:00:30,0 days 00:00:23,0 days 00:06:33,0 days 00:02:43
flash_amp,28959.0,32000.0,-9762.0,8842.0,-3236.0,-6218.0
flash_times_before,"[2018-06-09 10:32:03, 2018-06-09 10:33:15, 201...","[2018-06-12 09:41:01, 2018-06-12 09:43:52, 201...","[2018-06-25 20:07:04, 2018-06-25 20:07:25, 201...","[2018-06-30 21:15:42, 2018-06-30 21:22:19, 201...","[2018-07-18 18:25:31, 2018-07-18 18:27:59]",[2018-08-03 17:04:05]
flash_amps_before,"[32000.0, 27234.0, 15248.0, -21890.0, 8219.0, ...","[-6541.0, 32000.0, 32000.0]","[-8300.0, 6551.0, 3635.0, -10921.0, 2708.0, 12...","[-14272.0, -17061.0, -25394.0, 17278.0, -9041....","[1485.0, -3400.0]",[11566.0]
lat_sat,62.437027,59.131897,63.243748,51.697651,51.668144,46.436596
lon_sat,100.644447,105.562088,51.315292,132.650574,119.529045,122.079979
